<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Introduction to Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">2. Spark DataFrames and SQL</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 2.2 Lecture - Reading and Writing Data
### Duration - 12 Minutes

> *DataFrame I/O, schemas, data types, and Spark SQL*

This lecture covers the interfaces Spark provides for getting data into and out of DataFrames, how schemas and data types work, and how Spark SQL and the metastore fit in.

## Learning Objectives

**By the end of this lecture, you will be able to:**

- Use the `DataFrameReader` and `DataFrameWriter` interfaces for data I/O
- Define DataFrame schemas by inference, with `StructType` objects, and with DDL strings
- Identify common primitive and complex Spark data types, including the `VARIANT` type
- Explain how Spark SQL and metastores (Unity Catalog) relate to DataFrames

## A. DataFrameReader and DataFrameWriter

<p style="font-size: 1em; line-height: 1.6; color: #333">The DataFrame API provides two interfaces for data I/O.</p>

**`DataFrameReader`** (accessed as `spark.read`):

- Supports multiple formats (CSV, JSON, Parquet, ORC, Delta, and more)
- Provides schema inference

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CCC">df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">spark</span><span style="color: #67CDCC">.</span><span style="color: #CCC">read</span><span style="color: #67CDCC">.</span><span style="color: #CCC">format(</span><span style="color: #7EC699">"csv"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">option(</span><span style="color: #7EC699">"header"</span><span style="color: #CCC">,</span> <span style="color: #7EC699">"true"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">load(</span><span style="color: #7EC699">"/path/to/csv-dir"</span><span style="color: #CCC">)</span>
<span></span>
<span style="color: #999"># or using predefined format readers</span>
<span style="color: #CCC">df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">spark</span><span style="color: #67CDCC">.</span><span style="color: #CCC">read</span><span style="color: #67CDCC">.</span><span style="color: #CCC">csv(</span><span style="color: #7EC699">"/Volumes/catalog/schema/files/csv-dir"</span><span style="color: #CCC">)</span>
<span style="color: #CCC">df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">spark</span><span style="color: #67CDCC">.</span><span style="color: #CCC">read</span><span style="color: #67CDCC">.</span><span style="color: #CCC">parquet(</span><span style="color: #7EC699">"s3://mybucket/parquet-dir"</span><span style="color: #CCC">)</span></code></pre>

**`DataFrameWriter`** (accessed as `dataframe.write`):

- Flexible output formats and partitioning
- Supports various save modes (`overwrite`, `append`, and more)

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CCC">df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">write</span><span style="color: #67CDCC">.</span><span style="color: #CCC">format(</span><span style="color: #7EC699">"parquet"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">mode(</span><span style="color: #7EC699">"overwrite"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">save(</span><span style="color: #7EC699">"/path/to/output-dir"</span><span style="color: #CCC">)</span>
<span></span>
<span style="color: #999"># or using predefined format writers</span>
<span style="color: #CCC">df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">write</span><span style="color: #67CDCC">.</span><span style="color: #CCC">csv(</span><span style="color: #7EC699">"/Volumes/catalog/schema/files/csv-out"</span><span style="color: #CCC">)</span>
<span style="color: #CCC">df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">write</span><span style="color: #67CDCC">.</span><span style="color: #CCC">saveAsTable(</span><span style="color: #7EC699">"my_catalog.my_schema.my_table"</span><span style="color: #CCC">)</span></code></pre>



## B. DataFrame Schemas

<p style="font-size: 1em; line-height: 1.6; color: #333">Every DataFrame has a defined schema - the structure and data types of all its columns. A schema can be inferred from the data or explicitly specified (more efficient):</p>

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Defining a schema explicitly</span>
<span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.types </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">StructType,</span> <span style="color: #CCC">StructField,</span> <span style="color: #CCC">StringType,</span> <span style="color: #CCC">IntegerType</span>
<span style="color: #CCC">schema</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">StructType([</span>
    <span style="color: #CCC">StructField(</span><span style="color: #7EC699">"name"</span><span style="color: #CCC">,</span> <span style="color: #CCC">StringType()),</span>
    <span style="color: #CCC">StructField(</span><span style="color: #7EC699">"age"</span><span style="color: #CCC">,</span> <span style="color: #CCC">IntegerType())</span>
<span style="color: #CCC">])</span></code></pre>

- Self-describing formats like Parquet and Delta include schema information, so no inference is needed
- Use the `printSchema()` method to print out a DataFrame's schema

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#9888;&#65039;</span>
<div>
<strong style="color: #e65100; font-size: 1.1em">Schema Inference Costs</strong>
<p style="margin: 8px 0 0 0; color: #333">Inferring a schema requires an extra pass over the data and can guess wrong (for example reading a postal code as an integer). For production pipelines, define schemas explicitly.</p>
</div>
</div>
</div>

### DDL Schemas

Instead of defining schemas using `StructType` objects, you can use DDL (Data Definition Language) strings for more readable schema definitions:

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Defining a DDL schema</span>
<span style="color: #CCC">ddl_schema</span> <span style="color: #67CDCC">=</span> <span style="color: #7EC699">"name STRING NOT NULL, age INT, city STRING"</span>
<span></span>
<span style="color: #999"># Using the schema</span>
<span style="color: #CCC">df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">spark</span><span style="color: #67CDCC">.</span><span style="color: #CCC">read</span><span style="color: #67CDCC">.</span><span style="color: #CCC">csv(</span><span style="color: #7EC699">"/Volumes/catalog/schema/files/csv-dir"</span><span style="color: #CCC">,</span> <span style="color: #CCC">schema</span><span style="color: #67CDCC">=</span><span style="color: #CCC">ddl_schema)</span>
<span style="color: #CCC">df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">printSchema()</span></code></pre>



## C. DataFrame Data Types

Spark defines a rich set of primitive data types, exposed both as Python type objects and SQL type names:

| pyspark.sql.types | Spark SQL Type | Python equivalent |
|-------------------|----------------|-------------------|
| __`ByteType`__ | `TINYINT` | `int` |
| __`ShortType`__ | `SMALLINT` | `int` |
| __`IntegerType`__ | `INT` | `int` |
| __`LongType`__ | `BIGINT` | `int` |
| __`FloatType`__ | `FLOAT` | `float` |
| __`DoubleType`__ | `DOUBLE` | `float` |
| __`BooleanType`__ | `BOOLEAN` | `bool` |
| __`StringType`__ | `STRING` | `str` |
| __`BinaryType`__ | `BINARY` | `bytearray` |
| __`TimestampType`__ | `TIMESTAMP` | `datetime.datetime` |
| __`DateType`__ | `DATE` | `datetime.date` |

### Complex and Semi-Structured Data Types

Spark also supports complex types for nested data, plus the `VARIANT` type introduced in Spark 4:

| DataType | SQL Type | Python equivalent | Notes |
|----------|----------|-------------------|-------|
| __`ArrayType`__ | `ARRAY` | `list` (or `tuple`) | Ordered elements of the same type |
| __`MapType`__ | `MAP` | `dict` | Keys and values can have different types |
| __`StructType`__ | `STRUCT` | `tuple`, `dict`, `namedtuple` | Named fields with specified types |
| __`VariantType`__ | `VARIANT` | - | Open semi-structured values (such as JSON) stored in an optimized binary form; new in Spark 4 |

For more information see the [data types reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html).


<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128161;</span>
<div>
<strong style="color: #00695c; font-size: 1.1em">VARIANT for JSON</strong>
<p style="margin: 8px 0 0 0; color: #333">Before Spark 4, semi-structured JSON was typically stored as a <code>STRING</code> and parsed on every query. <code>VARIANT</code> stores the parsed structure in a binary form, so path expressions like <code>data:order.items[0].sku</code> are both more convenient and significantly faster.</p>
</div>
</div>
</div>

## D. Spark SQL

<p style="font-size: 1em; line-height: 1.6; color: #333">Spark SQL is the SQL interface to the same engine that runs DataFrames - the two are interchangeable.</p>

**DataFrame registration:**

- Temporary views: `createOrReplaceTempView()`
- Global temp views: `createGlobalTempView()`
- Registration enables SQL queries against DataFrames

**SQL query execution:**

- Use `spark.sql()` to run SQL statements from Python
- Results come back as DataFrames, so SQL and DataFrame code compose seamlessly
- Full SQL support including complex types, expressions, and the built-in functions library

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #CCC">df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">createOrReplaceTempView(</span><span style="color: #7EC699">"flights_temp"</span><span style="color: #CCC">)</span>
<span style="color: #CCC">result_df</span> <span style="color: #67CDCC">=</span> <span style="color: #CCC">spark</span><span style="color: #67CDCC">.</span><span style="color: #CCC">sql(</span><span style="color: #7EC699">"SELECT carrier, COUNT(*) AS flights FROM flights_temp GROUP BY carrier"</span><span style="color: #CCC">)</span></code></pre>



## E. A Primer on Metastores

<p style="font-size: 1em; line-height: 1.6; color: #333">A <strong>metastore</strong> provides the metadata management for Spark SQL. It is responsible for:</p>

- Defining table schemas and locations
- Defining table partitions
- Storing properties and information for objects including tables, views, and functions

<p style="font-size: 1em; line-height: 1.6; color: #333"><strong>Unity Catalog</strong> is the centralized metadata service provided for Databricks accounts and assigned to Databricks workspaces. It enables fine-grained security and governance across all data assets, organizing them in a three-level namespace: <code>catalog.schema.table</code>.</p>

## Summary

- `spark.read` (the `DataFrameReader`) and `df.write` (the `DataFrameWriter`) handle all DataFrame I/O
- Schemas can be inferred, defined with `StructType` objects, or defined with readable DDL strings - explicit is preferred
- Spark provides primitive, complex, and (new in Spark 4) `VARIANT` data types
- Spark SQL and the DataFrame API are two interfaces to the same engine; temp views bridge between them
- Unity Catalog is the metastore in Databricks, governing tables, views, volumes, and more

## Knowledge Check

Answer the questions below to check your understanding of this lecture.

<div style="font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif;width:100%;margin:10px 0;padding:24px;background:#f5f7fa;border-radius:12px;border:1px solid #e0e0e0;box-sizing:border-box">
<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px"><div><div style="font-size:1.4em;font-weight:600;color:#333">📝 Multiple Choice Quiz</div><div style="font-size:0.95em;color:#666;margin-top:4px">Knowledge Check</div></div><span style="padding:8px 16px;border-radius:20px;font-weight:600;font-size:0.9em;background:#e3f2fd;color:#1976d2">3 questions</span></div>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 1 of 3</div><div style="font-size:1.15em;color:#333;line-height:1.6">To create a DataFrame from various types of files, Spark provides a class called ______.</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">DataFrameCreator</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">DataFrameReader</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">DataFrameWriter</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">DataFrameFactory</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ B. DataFrameReader</div><div style="color:#333;margin-top:6px;line-height:1.5">The DataFrameReader, accessed via spark.read, reads files, tables, and external sources into DataFrames. The DataFrameWriter (df.write) goes the other way.</div></div></details>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 2 of 3</div><div style="font-size:1.15em;color:#333;line-height:1.6">Which of the following are valid ways to explicitly define a DataFrame schema?</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">Only StructType objects</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">Only DDL strings</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">Both StructType objects and DDL strings</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">Schemas can only be inferred from the data</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ C. Both StructType objects and DDL strings</div><div style="color:#333;margin-top:6px;line-height:1.5">A schema can be defined programmatically with StructType/StructField objects, or more readably with a DDL string such as 'name STRING, age INT'. Both are explicit alternatives to inference.</div></div></details>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 3 of 3</div><div style="font-size:1.15em;color:#333;line-height:1.6">Which data type, introduced in Spark 4, is designed for semi-structured data such as JSON?</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">ARRAY</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">STRUCT</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">BINARY</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">VARIANT</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ D. VARIANT</div><div style="color:#333;margin-top:6px;line-height:1.5">VARIANT stores open semi-structured values in an optimized binary encoding, replacing the older pattern of storing JSON as strings and parsing it on every query.</div></div></details>
</div>

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>